# Notebook 2 — Predict **peak memory (MB)** per compression config

**LightGBM, 10-value output.** For a single prompt this model predicts peak memory in MB (the CSV's `peak_memory_mb` column, read directly -- no unit conversion) under each of the 10 compression configurations (Baseline full precision, KVQuant 2/3/4-bit, H2O 20/40/60%, RocketKV 8/16/32x) — a length-10 vector.

**Protocol.** Identical to Notebook 1 (reshape → hold out 15% test → repeated stratified k-fold CV on 85% → refit → test once). Peak-memory values span a wide range across datasets, so targets are fit in log1p space and inverted; errors are reported in MB.

## Setup

In [ ]:
# Install deps (Colab-safe; no-op if already present). Add --break-system-packages
# locally if pip refuses to touch a managed environment.
!pip install -q lightgbm scikit-learn scipy pandas numpy joblib requests
import os, warnings
os.makedirs("artifacts", exist_ok=True)
warnings.filterwarnings("ignore", message="X does not have valid feature names")


## Engine — data loading, reshape, features

In [ ]:
# ===========================================================================
# ENGINE — shared across the latency / memory / correctness notebooks.
# Only the CONFIG block below changes between the three notebooks.
# ===========================================================================
import io, os, re, time, itertools
from pathlib import Path
import numpy as np, pandas as pd, requests

# ------------------------------- CONFIG ------------------------------------
TARGET   = "peak_memory_mb"    # the real per-prompt CSVs store this natively in MB
ARTIFACT = "reg_peak_memory_lightgbm.joblib"
# ---------------------------------------------------------------------------

RS = 42                    # global random seed — one split, reproducible everywhere
TEST_FRAC = 0.15           # 15% held out for the final test; 85% for train+val+CV

# The 10 compression configurations (Baseline = full precision, no compression).
# THIS is the output axis: every model emits a length-10 vector, one predicted
# value per configuration, for a single prompt.
CONFIGS = ["baseline",
           "kvquant_2bit", "kvquant_3bit", "kvquant_4bit",
           "h2o_20", "h2o_40", "h2o_60",
           "rocketkv_8x", "rocketkv_16x", "rocketkv_32x"]

# Map each config to the on-disk / on-repo filename stem.
FNAME = {"baseline": "kvquant_baseline_full_precision",
         "kvquant_2bit": "kvquant_2bit", "kvquant_3bit": "kvquant_3bit", "kvquant_4bit": "kvquant_4bit",
         "h2o_20": "h2o_budget_20pct", "h2o_40": "h2o_budget_40pct", "h2o_60": "h2o_budget_60pct",
         "rocketkv_8x": "rocketkv_ratio_8x", "rocketkv_16x": "rocketkv_ratio_16x", "rocketkv_32x": "rocketkv_ratio_32x"}

DATASETS = ["gsm8k", "arc_challenge", "hellaswag", "squad"]

# Load CSVs from Google Drive -- KVQuant_v3_Results is the authoritative
# source every implementation notebook actually writes to; the repo's
# 2048_sample_results2/ folder is just a checked-in COPY of it, kept as a
# fallback in case Drive isn't mounted (e.g. local Jupyter) or a file hasn't
# been re-copied there yet. Falls back further to a local ./Data checkout or
# raw GitHub as a last resort.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass          # not running in Colab (e.g. local Jupyter) -- fall through

# Where the trained model artifact gets saved -- this MUST be the same Drive
# folder evaluate_adaptive_compression() loads from, or that notebook keeps
# reading a stale artifact no matter how many times this one reruns.
MODEL_DIR = Path("/content/drive/MyDrive/KV_Cache_Models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RESULTS2_DIRS = ["2048_sample_results2", "../2048_sample_results2", "../../2048_sample_results2"]
DRIVE_DIR  = "/content/drive/MyDrive/KVQuant_v3_Results"
LOCAL_DIRS = [DRIVE_DIR] + RESULTS2_DIRS + ["Data", "../Data", "../../Data"]
REPO_RAW   = "https://raw.githubusercontent.com/yoshikodes/KVCacheCompression/main/2048_sample_results2"

def _local_dir():
    for d in LOCAL_DIRS:
        if os.path.isdir(d) and any(f.endswith("_per_prompt.csv") for f in os.listdir(d)):
            return d
    return None

_LOCAL = _local_dir()

def _read_csv_file(fn):
    """Read one CSV by exact filename from _LOCAL or REPO_RAW. Raises
    FileNotFoundError (local) or requests.HTTPError (remote, typically 404)
    if it doesn't exist there -- callers use that to fall back."""
    if _LOCAL:
        return pd.read_csv(os.path.join(_LOCAL, fn))
    r = requests.get(f"{REPO_RAW}/{fn}", timeout=60)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

def _read(cfg, ds):
    """Read one per-prompt CSV; standardize the index column name to 'idx'.
    Also standardize the prompt column to 'prompt' -- some per-prompt CSVs
    (e.g. ones carrying a RULER-style prefilled-prompt column) store it as
    'full_prompt' instead. Some KVQuant runs saved HellaSwag (and RULER) as
    two checkpoint-safety batches instead of one combined file -- if the
    plain filename isn't found, fall back to batch1+batch2 and concatenate.
    IMPORTANT: each batch's raw index column is LOCAL to that batch (both
    restart at 0!), so naively concatenating gives every idx value two rows
    -- silently turning every downstream merge in build_wide() into a
    many-to-many join that MULTIPLIES the row count instead of adding to it
    (e.g. HellaSwag would balloon from 1024 rows to 16384). Offset batch2's
    raw index by batch1's row count so every value is globally unique before
    concatenating -- this matches how the batches were actually sliced
    (items[:N] then items[N:2N])."""
    fn = f"{FNAME[cfg]}_{ds}_per_prompt.csv"
    try:
        df = _read_csv_file(fn)
    except (FileNotFoundError, requests.exceptions.HTTPError):
        parts = []
        offset = 0
        for suffix in ("_batch1", "_batch2"):
            part = _read_csv_file(f"{FNAME[cfg]}_{ds}{suffix}_per_prompt.csv")
            idxc = "question_index" if "question_index" in part.columns else "item_index"
            part[idxc] = part[idxc] + offset
            offset = part[idxc].max() + 1
            parts.append(part)
        df = pd.concat(parts, ignore_index=True)
    idxc = "question_index" if "question_index" in df.columns else "item_index"
    df = df.rename(columns={idxc: "idx"})
    if "prompt" not in df.columns and "full_prompt" in df.columns:
        df = df.rename(columns={"full_prompt": "prompt"})
    return df

print("Data source:", _LOCAL if _LOCAL else REPO_RAW)

# ===========================================================================
# BUILD THE WIDE TABLE: one row per (dataset, prompt); 10 target columns.
# The stored "prompt" column IS the exact text that was fed to the model --
# already fully prefilled (fewshot prefix, answer choices, everything) -- so
# no reconstruction is needed; it's taken as-is from a single reference
# config. The bare question is identical across all 10 configs for a given
# index (the eval order is seed-42 stable across methods), which is what
# makes using a single reference config safe.
# ===========================================================================
def build_wide():
    frames = []
    for ds in DATASETS:
        ref = _read("kvquant_2bit", ds)[["idx", "prompt"]]
        base = pd.DataFrame({"dataset": ds, "idx": ref["idx"].values, "model_input": list(ref["prompt"])})
        for cfg in CONFIGS:                          # attach each config's target column
            d = _read(cfg, ds)[["idx", TARGET]].rename(columns={TARGET: cfg})
            base = base.merge(d, on="idx", how="inner")
        frames.append(base)
    wide = pd.concat(frames, ignore_index=True)
    assert int(wide[CONFIGS].isna().sum().sum()) == 0, "unexpected NaN targets"
    return wide

# ===========================================================================
# FEATURES: cheap numeric descriptions of the prompt text, plus a
# one-hot of the dataset. NOTE: no feature encodes the compression config —
# the config is the OUTPUT axis, so a single prompt maps to one feature row and
# ten predicted values.
# ===========================================================================
def _feats(p):
    s = str(p); t = s.split(); nt = max(len(t), 1); nc = max(len(s), 1)
    return {
        "n_char":      len(s),
        "n_tok":       len(t),
        "ttr":         len(set(t)) / nt,                                   # type-token ratio
        "digit_ratio": sum(c.isdigit() for c in s) / nc,
        "punct_ratio": sum(not c.isalnum() and not c.isspace() for c in s) / nc,
        "upper_ratio": sum(c.isupper() for c in s) / nc,
        "avg_tok":     nc / nt,                                            # mean word length
        "num_count":   len(re.findall(r"\d+", s)),
        "has_q":       int("?" in s),
        "n_newline":   s.count("\n"),
    }

DS_DUMMY_COLS = [f"ds_{d}" for d in DATASETS]        # fixed column order for the one-hot

def fmat(df):
    X = pd.DataFrame([_feats(p) for p in df["model_input"]])
    dummies = pd.get_dummies(df["dataset"], prefix="ds").reindex(columns=DS_DUMMY_COLS, fill_value=0)
    return pd.concat([X.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

# Target matrix (n_prompts, 10), column order == CONFIGS.
def Ymat(df):
    return df[CONFIGS].to_numpy(dtype=float)


## Load, patch prompts, and hold out the test set once

In [ ]:
# ---------------------------------------------------------------------------
# DIAGNOSTIC: confirm every CSV has a unique idx per row before merging.
# build_wide() does an inner merge on "idx" for each of the 10 configs; if
# idx is ever duplicated within one CSV (e.g. a batch1/batch2 concatenation
# gone wrong), the merge silently becomes many-to-many and the row count
# multiplies instead of staying at one row per prompt.
# ---------------------------------------------------------------------------
for ds in DATASETS:
    print(f"\n=== {ds} ===")
    for cfg in CONFIGS:
        d = _read(cfg, ds)
        print(
            cfg,
            "| rows:", len(d),
            "| unique idx:", d["idx"].nunique(),
            "| duplicated rows:", d["idx"].duplicated().sum()
        )


In [ ]:
from sklearn.model_selection import train_test_split

# Build the wide table (attaches the 10 targets).
wide = build_wide()
print("wide table:", wide.shape, "| rows per dataset:", wide["dataset"].value_counts().to_dict())

# --------------------------------------------------------------------------
# STEP 1 — hold out a TEST set ONCE, stratified by dataset so the 15% test has
# the same gsm8k/arc/hellaswag mix as the whole. The test set is NOT looked at
# during tuning; it is touched exactly once, at the very end.
# --------------------------------------------------------------------------
tr_parts, te_parts = [], []
for ds, g in wide.groupby("dataset"):
    a, b = train_test_split(g, test_size=TEST_FRAC, random_state=RS, shuffle=True)
    tr_parts.append(a); te_parts.append(b)

trainval_df = pd.concat(tr_parts).sample(frac=1, random_state=RS).reset_index(drop=True)
test_df     = pd.concat(te_parts).sample(frac=1, random_state=RS).reset_index(drop=True)

print("train+val:", len(trainval_df), "| test (held out):", len(test_df))
print("per-dataset train+val:", trainval_df["dataset"].value_counts().to_dict())

# Peek at one patched prompt per dataset so the patching is visible.
for ds in DATASETS:
    ex = wide[wide.dataset == ds]["model_input"].iloc[0]
    print(f"\n--- model_input | {ds} (first {180} chars) ---\n{ex[:180]!r}")


## Targets

In [ ]:
# --------------------------------------------------------------------------
# Targets are strictly positive and span a wide range (seconds / MB across very
# different datasets), so we fit in log1p space and invert with expm1 for all
# reported predictions and metrics — errors stay in the ORIGINAL units.
# --------------------------------------------------------------------------
from sklearn.metrics import mean_absolute_error, r2_score

def to_train_space(Y):   return np.log1p(Y)
def to_orig_space(Yhat): return np.expm1(Yhat)

# Quick sanity print of the 10-way target ranges.
_Y = Ymat(trainval_df)
print("target:", TARGET)
for j, c in enumerate(CONFIGS):
    print(f"  {c:14s} min={_Y[:,j].min():.4g}  med={np.median(_Y[:,j]):.4g}  max={_Y[:,j].max():.4g}")


## Hyperparameter search — repeated stratified k-fold CV (train+val only)

In [ ]:
import lightgbm as lgb
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import RepeatedStratifiedKFold

# ==========================================================================
# STEP 2 -- choose hyperparameters with REPEATED k-fold CV on train+val ONLY,
# INDEPENDENTLY per config. Each of the 10 configs runs its own CV sweep and
# picks its own best hyperparameters -- they are NOT forced to share one
# setting. (Sharing one setting is risky in general: if any one config's target
# happens to have a much larger raw-MB scale than the others, a selection
# metric that averages an unnormalized error measure across all 10 configs
# would effectively be decided by that one config alone, regardless of what
# the other 9 configs needed.) The grid is also
# wider than before, and learning_rate is now searched instead of fixed.
# Repeated k-fold = run k-fold N_REPEATS times, each with a fresh random
# partition; averaging over all folds*repeats gives a stabler estimate and a
# real error bar. Folds are stratified by dataset to keep the regime mix.
# The test set is NOT touched here.
# ==========================================================================
NUM_LEAVES_GRID    = [15, 31, 63, 127]
MIN_CHILD_GRID     = [10, 20, 50, 100]
N_ESTIMATORS_GRID  = [200, 300, 600]
LEARNING_RATE_GRID = [0.03, 0.1]
CANDIDATES = [{"num_leaves": nl, "min_child_samples": mc, "n_estimators": ne, "learning_rate": lr}
              for nl, mc, ne, lr in itertools.product(
                  NUM_LEAVES_GRID, MIN_CHILD_GRID, N_ESTIMATORS_GRID, LEARNING_RATE_GRID)]

FIXED = dict(subsample=0.8, colsample_bytree=0.8, random_state=RS, verbose=-1, n_jobs=-1)

N_FOLDS, N_REPEATS = 5, 3
print(f"{len(CANDIDATES)} candidates x {N_FOLDS}-fold x {N_REPEATS} repeats "
      f"x {len(CONFIGS)} outputs = {len(CANDIDATES)*N_FOLDS*N_REPEATS*len(CONFIGS)} model fits\n")


class PerTargetLGBMRegressor(BaseEstimator, RegressorMixin):
    """ONE fitted object wrapping N independently-tuned LGBMRegressor models
    -- one per config -- each using hyperparameters chosen by that config's
    OWN cross-validation (the sweep below) instead of every config sharing a
    single hyperparameter setting. Still ONE fitted object with one
    .fit()/.predict() call, not a hand-managed Python list -- this keeps
    MultiOutputRegressor's external contract without its "one shared config
    for every target" limitation, which is exactly what MultiOutputRegressor
    can't do (it only knows how to clone a single estimator/config)."""
    def __init__(self, per_target_params):
        self.per_target_params = per_target_params   # list of dicts, len == len(CONFIGS)

    def fit(self, X, Y):
        self.estimators_ = [lgb.LGBMRegressor(**params).fit(X, Y[:, j])
                             for j, params in enumerate(self.per_target_params)]
        return self

    def predict(self, X):
        return np.column_stack([est.predict(X) for est in self.estimators_])


def fit_9(per_target_params, Xtr, Ytr_log):
    model = PerTargetLGBMRegressor(per_target_params)
    model.fit(Xtr, Ytr_log)
    return model

def predict_9(model, X):
    return to_orig_space(model.predict(X))

def cv_score_per_target(cfg):
    """Returns (n_fold_repeats, len(CONFIGS)) arrays of per-config MAE and R2
    for this ONE candidate -- NOT averaged across configs, so each config's
    own best hyperparameters can be picked independently below."""
    rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=RS)
    strat = trainval_df["dataset"].values
    mae_rows, r2_rows = [], []
    for tr_idx, va_idx in rskf.split(trainval_df, strat):
        tr, va = trainval_df.iloc[tr_idx], trainval_df.iloc[va_idx]
        Xtr, Xva = fmat(tr).values, fmat(va).values
        Ytr_log = to_train_space(Ymat(tr))
        pred = to_orig_space(np.column_stack([
            lgb.LGBMRegressor(**cfg, **FIXED).fit(Xtr, Ytr_log[:, j]).predict(Xva)
            for j in range(len(CONFIGS))
        ]))
        yt = Ymat(va)
        mae_rows.append([mean_absolute_error(yt[:, j], pred[:, j]) for j in range(len(CONFIGS))])
        r2_rows.append([r2_score(yt[:, j], pred[:, j]) for j in range(len(CONFIGS))])
    return np.array(mae_rows), np.array(r2_rows)

mean_mae = np.zeros((len(CANDIDATES), len(CONFIGS)))
mean_r2  = np.zeros((len(CANDIDATES), len(CONFIGS)))
for i, cfg in enumerate(CANDIDATES):
    mae_mat, r2_mat = cv_score_per_target(cfg)
    mean_mae[i] = mae_mat.mean(axis=0)
    mean_r2[i]  = r2_mat.mean(axis=0)
    worst, best = CONFIGS[mean_r2[i].argmin()], CONFIGS[mean_r2[i].argmax()]
    print(f"[{i+1:>3}/{len(CANDIDATES)}] {str(cfg):<70} "
          f"R2 mean={mean_r2[i].mean():.4f}  (worst {worst}={mean_r2[i].min():.4f}, best {best}={mean_r2[i].max():.4f})")

# Select each config's OWN best hyperparameters independently, by highest
# mean CV R^2. R^2 is scale-invariant (normalized by each config's own
# variance), so no single config's raw-MB scale can dominate another
# config's selection the way an unweighted mean of raw MAE across configs
# would.
best_idx_per_config = mean_r2.argmax(axis=0)
best_cfg_per_config = {c: {**CANDIDATES[best_idx_per_config[j]], **FIXED} for j, c in enumerate(CONFIGS)}

print("\nBest hyperparameters per config (selected independently, by mean CV R^2):")
for j, c in enumerate(CONFIGS):
    idx = best_idx_per_config[j]
    print(f"  {c:14s} {CANDIDATES[idx]}  CV R2={mean_r2[idx, j]:.4f}  CV MAE={mean_mae[idx, j]:.4f}")


## Final evaluation — test set touched once

In [ ]:
# ==========================================================================
# STEP 3 — final fit on ALL train+val with the CV-chosen config; evaluate the
# held-out TEST set exactly once. Save the 10 models + metadata.
# ==========================================================================
import joblib

Xtv, Xte = fmat(trainval_df).values, fmat(test_df).values
Ytv, Yte = Ymat(trainval_df), Ymat(test_df)

per_target_params = [best_cfg_per_config[c] for c in CONFIGS]
final_model = fit_9(per_target_params, Xtv, to_train_space(Ytv))
pred_tv = predict_9(final_model, Xtv)
pred_te = predict_9(final_model, Xte)

payload = {"task": "regression", "target": TARGET, "configs": CONFIGS,
           "feature_cols": list(fmat(trainval_df).columns),
           "model": final_model, "config": best_cfg_per_config,
           "target_space": "log1p", "seed": RS}
joblib.dump(payload, MODEL_DIR / ARTIFACT)
print("saved", MODEL_DIR / ARTIFACT)
print("hyperparameters per config:")
for c in CONFIGS:
    print(f"  {c:14s} {best_cfg_per_config[c]}")

def per_config_table(Yt, Pt, Yv, Pv):
    rows = []
    for j, c in enumerate(CONFIGS):
        rows.append({"config": c,
                     "MAE_trainval": mean_absolute_error(Yv[:, j], Pv[:, j]),
                     "R2_trainval":  r2_score(Yv[:, j], Pv[:, j]),
                     "MAE_test":     mean_absolute_error(Yt[:, j], Pt[:, j]),
                     "R2_test":      r2_score(Yt[:, j], Pt[:, j])})
    tab = pd.DataFrame(rows)
    tab.loc[len(tab)] = {"config": "MEAN",
        "MAE_trainval": tab.MAE_trainval.mean(), "R2_trainval": tab.R2_trainval.mean(),
        "MAE_test": tab.MAE_test.mean(),         "R2_test": tab.R2_test.mean()}
    return tab

tab = per_config_table(Yte, pred_te, Ytv, pred_tv)
print("\n=== PER-CONFIG RESULTS (mean over the 10-vector at the bottom) ===")
print(tab.round(4).to_string(index=False))
print(f"\ngeneralization gap (test MAE - trainval MAE): "
      f"{tab.iloc[-1].MAE_test - tab.iloc[-1].MAE_trainval:.4f}  (small = good)")


## Inference — the length-10 output vector

In [ ]:
# ==========================================================================
# INFERENCE HELPER — one prompt in, a length-10 vector out (original units).
# The 10 entries line up with CONFIGS.
# ==========================================================================
def predict_vector(prompt_text, dataset):
    """dataset in {'gsm8k','arc_challenge','hellaswag'} — used only for the one-hot."""
    row = pd.DataFrame({"model_input": [str(prompt_text)], "dataset": [dataset]})
    X = fmat(row).values
    vec = predict_9(final_model, X)[0]
    return dict(zip(CONFIGS, vec))

# demo on the first held-out test prompt
_ex = test_df.iloc[0]
print("dataset:", _ex["dataset"], "| TARGET =", TARGET)
pred_vec = predict_vector(_ex["model_input"], _ex["dataset"])
for c in CONFIGS:
    print(f"  {c:14s} predicted={pred_vec[c]:.4g}   actual={_ex[c]:.4g}")


## How to read this

- The **CV table** is where hyperparameters are chosen — by mean validation score across all folds *and* repeats, with the std as an error bar. If two configs are within ~1 std, prefer the simpler (more regularized) one.
- The **per-config table** reports the held-out test result for each of the 10 configurations, with a **MEAN** row summarizing the length-10 output vector. The test set influenced neither the features nor the hyperparameters, so it is an honest estimate.
- `predict_vector(prompt, dataset)` shows the end use: one prompt in, the length-10 vector out.
- The saved `/content/drive/MyDrive/KV_Cache_Models/*.joblib` bundles the model (one `PerTargetLGBMRegressor` object emitting all 10 values, each with its own independently-tuned hyperparameters), the feature-column order, the chosen config-per-target dict and the target space, so predictions can be reproduced elsewhere.